# Decision Tree Baseline: Job Satisfaction

## Overview

This notebook implements a baseline Decision Tree classifier to predict binary job satisfaction (`satjob`) using the cleaned GSS 2024 dataset combined with O*NET work values and occupation features.

## Data Preparation & Preprocessing

* **Target Binarization:** Converted the 4-point `satjob` scale into a binary outcome where original responses 2 through 4 map to `1` (Satisfied/Neutral) and response 1 maps to `0` (Dissatisfied).
* **Missing Values:** Dropped rows with missing target values, yielding a final working dataset of **2,782 records** across 29 features.
* **Categorical Encoding:** Applied one-hot encoding (`pd.get_dummies` with `drop_first=True`) across categorical attributes (employment status, sex, race, education, marital status, region, union membership, etc.), expanding the feature matrix to **45 columns**.
* **Train/Test Split:** Split the data using an 80/20 train-test ratio (`random_state=42`).

## Model & Hyperparameter Tuning

We trained a `DecisionTreeClassifier` and used `GridSearchCV` to tune key hyperparameters.

* **Evaluated Parameters:**
* `criterion`: `['gini', 'entropy']`
* `max_depth`: `[None, 3, 5, 10]`
* `min_samples_split`: `[2, 5, 10]`
* `min_samples_leaf`: `[1, 2, 4]`
* `class_weight`: `[None, 'balanced']`


* **Optimal Configuration Found:**
* `criterion`: `gini`
* `max_depth`: `5`
* `min_samples_split`: `2`
* `min_samples_leaf`: `2`
* `class_weight`: `None`

## Results & Performance

* **GridSearchCV Best CV Score:** 71.20%
* **Test Set Accuracy:** 72.18%

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

import matplotlib.pyplot as plt

In [3]:
data_path = "gss2024_onet_cleaned.csv"
df = pd.read_csv(data_path)
print(df.shape)
print(df.columns)

df.head()

(3986, 29)
Index(['satjob', 'occ10', 'indus10', 'wrkstat', 'wrkslf', 'prestg10', 'sei10',
       'realrinc', 'rincom16', 'hrs1', 'weekswrk', 'age', 'sex', 'race',
       'hispanic', 'born', 'educ', 'degree', 'marital', 'childs', 'region',
       'union1', 'Achievement', 'Independence', 'Recognition', 'Relationships',
       'Support', 'Working Conditions', 'job_zone'],
      dtype='object')


,satjob,occ10,indus10,wrkstat,wrkslf,prestg10,sei10,realrinc,rincom16,hrs1,...,childs,region,union1,Achievement,Independence,Recognition,Relationships,Support,Working Conditions,job_zone
0,2.0,230.0,7860.0,1.0,2.0,59.0,75.5,29947.5,21.0,43.0,...,2.0,1.0,4.0,5.7525,6.08250,5.250,6.2500,3.915,5.33250,4.5
1,NaN,800.0,7280.0,5.0,2.0,60.0,76.3,16335.0,18.0,40.0,...,0.0,1.0,4.0,5.0000,5.00000,4.670,4.6700,4.670,4.50000,4.0
2,NaN,430.0,3570.0,5.0,2.0,39.0,67.7,16335.0,18.0,40.0,...,0.0,1.0,2.0,5.5000,5.29125,4.875,4.8325,4.375,5.04125,4.0
3,1.0,4760.0,4880.0,2.0,2.0,31.0,39.7,16335.0,18.0,20.0,...,0.0,1.0,4.0,3.0000,3.00000,2.670,5.0000,4.000,2.33000,2.0
4,NaN,5860.0,7860.0,5.0,2.0,32.0,35.8,16335.0,18.0,40.0,...,3.0,1.0,4.0,2.3300,2.67000,2.330,5.6700,4.330,2.50000,2.0


# Features description

## Target & Demographic/Employment Features (GSS)

**3986 entries**

- satjob: Respondent's self-reported job satisfaction level (Target Variable)

- occ10: Standardized census code used for crosswalk matching

- indus10: Standardized textual title for the census occupation

- wrkstat: Labor force status (1 = Working full time, 2 = Working part time, 3 = With a job, but tutorially/temporarily not at work (e.g., illness, vacation, strike), 4 = Unemployed, laid off, 5 = Retired, 6 = In school, 7 = Keeping house)

- wrkslf: Self-employment status (1 = self-employed, 2 = working for someone else)

- prestg10: Standardized occupational prestige score (2010 basis)

- sei10: Socioeconomic Index (SEI) score for the occupation (2010 basis)

- realrinc: Respondent's real family income

- rincom16: Respondent's individual income category/score

- hrs1: Number of hours worked per week

- weekswrk: Total number of weeks worked in the past year

- age: Age of the respondent

- sex: Sex of the respondent (1 = Male, 2 = Female)

- race: Race of the respondent (1 = White, 2 = Black, 3 = Other)

- hispanic: Hispanic origin/ethnicity identification (1 = Mexican, Mexican American, Chicano/Chicana, 2 = Puerto Rican, 3 = Cuban, 4 = Other, 5 = Not Hispanic)

- born: Country of birth (1 = Born in the United States, 2 = Not Born in the United States) 

- educ: Highest number of years of school completed   

- degree: Highest educational degree attained (0 = Less than high school, 1 = High school diploma, 2 = Associate / Junior college degree, 3 = Bachelor's degree, 4 = Graduate degree)

- marital: Marital status of the respondent (1 = Married, 2 = Widowed, 3 = Divorced, 4 = Separated, 5 = Never Married)

- childs: Total number of children the respondent has   

- region: US Census region where the interview took place (1 = Northeast (New England & Middle Atlantic), 2 = Midwest / North Central (East North Central & West North Central), 3 = South (South Atlantic, East South Central, & West South Central), 4 = West (Mountain & Pacific))

- union1: Union membership status 1 = Respondent belongs, 2 = Spouse or partner belongs, 3 = Both belong, 4 = Neither belongs

## O*NET Work Values & Job Characteristics Features
- Achievement: O*NET work value score measuring the importance of accomplishment and results   

- Independence: O*NET work value score measuring autonomy and independent decision-making   

- Recognition: O*NET work value score measuring potential for advancement and praise   

- Relationships: O*NET work value score measuring friendly environments and co-worker support   

- Support: O*NET work value score measuring supportive management and company policies   

- Working Conditions: O*NET work value score measuring physical environment comfort and safety   

- job_zone: O*NET classification score of the education, experience, and training required for the job (1 = Little or No Preparation Needed, 2 = Some Preparation Needed, 3 = Medium Preparation Needed, 4 = Considerable Preparation Needed, 5 = Extensive Preparation Needed)

In [6]:
print(df.isnull().sum())

df.fillna(df.median(numeric_only=True), inplace=True)

satjob                1204
occ10                    0
indus10                  0
wrkstat                  0
wrkslf                   0
prestg10                 0
sei10                    0
realrinc                 0
rincom16                 0
hrs1                     0
weekswrk                 0
age                      0
sex                      0
race                     0
hispanic                 0
born                     0
educ                     0
degree                   0
marital                  0
childs                   0
region                   0
union1                   0
Achievement              0
Independence             0
Recognition              0
Relationships            0
Support                  0
Working Conditions       0
job_zone                 0
dtype: int64


In [8]:
#Binarization
df['satjob'] = df['satjob'].replace({1:0, 2:1, 3:1, 4:1})

In [10]:
df = df.dropna(subset=['satjob'])
print(df.shape)

X = df.drop(columns=['satjob'])
y = df['satjob']

(3986, 29)


In [12]:
categorical_cols = ['wrkstat', 'wrkslf', 'sex', 'race', 'hispanic', 'born', 
    'degree', 'marital', 'region', 'union1']

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(X.columns)


Index(['occ10', 'indus10', 'prestg10', 'sei10', 'realrinc', 'rincom16', 'hrs1',
       'weekswrk', 'age', 'educ', 'childs', 'Achievement', 'Independence',
       'Recognition', 'Relationships', 'Support', 'Working Conditions',
       'job_zone', 'wrkstat_2.0', 'wrkstat_3.0', 'wrkstat_4.0', 'wrkstat_5.0',
       'wrkstat_6.0', 'wrkstat_7.0', 'wrkstat_8.0', 'wrkslf_2.0', 'sex_2.0',
       'race_2.0', 'race_3.0', 'hispanic_2.0', 'hispanic_3.0', 'hispanic_4.0',
       'hispanic_5.0', 'born_2.0', 'degree_1.0', 'degree_2.0', 'degree_3.0',
       'degree_4.0', 'marital_2.0', 'marital_3.0', 'marital_4.0',
       'marital_5.0', 'region_2.0', 'region_3.0', 'region_4.0', 'union1_2.0',
       'union1_3.0', 'union1_4.0'],
      dtype='object')


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# classifier = DecisionTreeClassifier(random_state=42, max_depth=5)
# classifier.fit(X_train, y_train)

# y_pred = classifier.predict(X_test)
# print("Model Accuracy:", accuracy_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [16]:
param_grid = {
    'max_depth': [3, 5, 8, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

dt = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=dt, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1           
)

grid_search.fit(X_train, y_train)

print("Best Parameters Found:")
print(grid_search.best_params_)
print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

best_classifier = grid_search.best_estimator_
y_pred = best_classifier.predict(X_test)
print("\nTest Set Accuracy with Best Model:", accuracy_score(y_test, y_pred))

Best Parameters Found:
{'class_weight': None, 'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2}

Best Cross-Validation Score: 0.7120

Test Set Accuracy with Best Model: 0.7218045112781954


#### Notes on Training Process

We dropped rows with missing target values, leaving **2,782 records**.

#### Binarization Pivot

Shifted from a multiclass problem to a binary target (satisfied vs. not satisfied) due to sparse representation and data gaps in ratings 3 and 4 (as highlighted by the classification reports).

* **Initial Threshold Plan:** Grouped ratings 1 & 2 as `1` and the rest as `0` (yielding ~84% accuracy).
* **Adjusted Threshold:** Revised based on guidance to group *only* "very satisfied" as `1` and the other three categories as `0` (yielding ~72% accuracy for a stricter definition of satisfaction).

#### Prior Multiclass Iterations (Pre-Binarization)

* **1st Iteration:** Standard decision tree (`max_depth=5`) yielded **48% accuracy**, with categories 3 and 4 showing `0.0` metrics due to poor class separation.
* **2nd Iteration:** Added `class_weight='balanced'`, which dropped overall performance to **29.6%**.
* **3rd Iteration:** Increased depth to `max_depth=8` with balanced weights, resulting in **30% accuracy**.
* **4th Iteration:** Removed class weights at `max_depth=8`, bumping accuracy to **44.8%** before pivoting to binary classification.